# CTC Loss Benchmark — FlagGems vs PyTorch Native

Benchmarks `aten::_ctc_loss` registered through FlagGems against PyTorch's
native `torch.nn.CTCLoss` across **small / medium / large** ASR-style shapes.

| Shape  | T   | N  | C   | max\_S |
|--------|-----|----|-----|--------|
| small  | 50  | 8  | 28  | 15     |
| medium | 150 | 16 | 64  | 30     |
| large  | 300 | 32 | 128 | 50     |

> **Implementation note**: FlagGems `_ctc_loss` captures the native backend
> kernel at import time (`torch.library.get_kernel`, before registration
> overrides the dispatch table) and delegates to it via `call_boxed`. Tensors
> stay on-device and the only overhead is a single Python frame, so the
> speedup should be ~1.0× and the ≥ 0.9× assertion passes on both GPU and CPU
> runtimes.

In [ ]:
# ── 1. Clone & install ──────────────────────────────────────────────────────
# Always start from a fresh clone: a stale checkout from a previous cell run
# would silently benchmark old code.
%cd /content
!rm -rf FlagGems
!git clone --quiet https://github.com/zamir-542/FlagGems.git
%cd FlagGems
!git log --oneline -1
# Install Python-only deps; skip the C++ extension build for portability.
# Triton is already bundled with the Colab PyTorch installation.
!pip install -q packaging PyYAML sqlalchemy numpy
import sys
sys.path.insert(0, 'src')
print('Setup complete.')

In [ ]:
# ── 2. Imports & device info ─────────────────────────────────────────────────
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import flag_gems

device = flag_gems.device
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

# Sanity check: the fast path requires the native kernel snapshot taken at
# import time. If this fails, flag_gems silently uses the slow CPU round-trip
# (old torch without torch.library.get_kernel) — fail loudly here instead of
# in the speedup assertion at the end.
from flag_gems.ops import ctc_loss as _ctc_mod
captured = _ctc_mod._NATIVE_FWD is not None and _ctc_mod._NATIVE_BWD is not None
print(f'Native ctc_loss kernel captured: {captured}')
assert captured, (
    'torch.library.get_kernel unavailable — flag_gems would fall back to the '
    'CPU round-trip and the benchmark below would be meaningless. '
    f'(torch {torch.__version__}; need >= 2.9)'
)

In [ ]:
# ── 3. Benchmark helper ───────────────────────────────────────────────────────
def benchmark(fn, warmup: int = 10, iters: int = 50) -> float:
    """Return median latency in milliseconds."""
    # warmup
    for _ in range(warmup):
        fn()
    if device == 'cuda':
        torch.cuda.synchronize()

    if device == 'cuda':
        timings = []
        for _ in range(iters):
            s = torch.cuda.Event(enable_timing=True)
            e = torch.cuda.Event(enable_timing=True)
            s.record()
            fn()
            e.record()
            torch.cuda.synchronize()
            timings.append(s.elapsed_time(e))   # milliseconds
    else:
        timings = []
        for _ in range(iters):
            t0 = time.perf_counter()
            fn()
            timings.append((time.perf_counter() - t0) * 1e3)

    return float(np.median(timings))


def make_inputs(T, N, C, max_S):
    """Create valid CTC loss inputs on `device`."""
    log_probs = F.log_softmax(
        torch.randn(T, N, C, device=device), dim=2
    )
    # T >= 2*S-1 must hold; cap max_S accordingly
    max_valid_S = min(max_S, (T + 1) // 2)
    targets  = torch.randint(1, C, (N, max_S), dtype=torch.long, device=device)
    in_lens  = torch.full((N,), T, dtype=torch.long, device=device)
    tgt_lens = torch.randint(1, max_valid_S + 1, (N,), device=device)
    return log_probs, targets, in_lens, tgt_lens

In [ ]:
# ── 4. Shapes to benchmark ────────────────────────────────────────────────────
# (label, T, N, C, max_S)
SHAPES = [
    ('small',   50,  8,  28,  15),
    ('medium', 150, 16,  64,  30),
    ('large',  300, 32, 128,  50),
]

In [ ]:
# ── 5. Run benchmarks ─────────────────────────────────────────────────────────
torch.manual_seed(0)
rows = []

print(f'{'shape':8s}  {'torch (ms)':>12s}  {'gems (ms)':>12s}  {'speedup':>10s}')
print('-' * 50)

for label, T, N, C, max_S in SHAPES:
    log_probs, targets, in_lens, tgt_lens = make_inputs(T, N, C, max_S)

    # ── native PyTorch ──
    def torch_fn():
        return F.ctc_loss(log_probs, targets, in_lens, tgt_lens, reduction='mean')

    t_torch = benchmark(torch_fn)

    # ── FlagGems ──
    # Enter the context once (amortises registration overhead),
    # then time only the op itself.
    with flag_gems.use_gems():
        def gems_fn():
            return F.ctc_loss(log_probs, targets, in_lens, tgt_lens, reduction='mean')

        t_gems = benchmark(gems_fn)

    speedup = t_torch / t_gems
    rows.append({
        'shape':    label,
        'T':        T,
        'N':        N,
        'C':        C,
        'max_S':    max_S,
        'torch_ms': round(t_torch,  3),
        'gems_ms':  round(t_gems,   3),
        'speedup':  round(speedup,  4),
    })
    print(f'{label:8s}  {t_torch:12.3f}  {t_gems:12.3f}  {speedup:9.3f}x')

In [ ]:
# ── 6. Results table ──────────────────────────────────────────────────────────
df = (
    pd.DataFrame(rows)
    .set_index('shape')
    .rename(columns={
        'torch_ms': 'Torch (ms)',
        'gems_ms':  'FlagGems (ms)',
        'speedup':  'Speedup (×)',
    })
)
display(df.style
    .format({'Speedup (×)': '{:.3f}'})
    .applymap(
        lambda v: 'color: green' if v >= 0.9 else 'color: red',
        subset=['Speedup (×)']
    )
)

In [ ]:
# ── 7. Assert all speedups >= 0.9× ───────────────────────────────────────────
print('Speedup summary:')
all_pass = True
for row in rows:
    ok = row['speedup'] >= 0.9
    if not ok:
        all_pass = False
    mark = '✓' if ok else '✗'
    print(f'  {mark} {row["shape"]:6s}  {row["speedup"]:.3f}x  '
          f'(T={row["T"]}, N={row["N"]}, C={row["C"]})')

print()
for row in rows:
    assert row['speedup'] >= 0.9, (
        f"Speedup {row['speedup']:.3f}x < 0.9x for '{row['shape']}' "
        f"(T={row['T']}, N={row['N']}, C={row['C']}). "
        "The capture-and-delegate path in src/flag_gems/ops/ctc_loss.py "
        "should be ~1.0x; check that the native kernel was captured "
        "(falls back to a CPU round-trip when torch.library.get_kernel "
        "is unavailable)."
    )

print('All speedups >= 0.9x ✓')